# Vector Search Playground

Three storage backends, one embedder, one dataset — to feel the difference between vector-search engines without changing the data or the model.

**What's compared**

| Engine | Where it lives | When you'd reach for it |
|---|---|---|
| In-memory NumPy | A matrix in RAM | Prototyping, small corpora, notebooks |
| sqlite-vec | A single `.db` file | Embedded apps, edge devices, simple deployments |
| pgvector | A running Postgres | Real products, large corpora, hybrid SQL + vector |

All three use the same ONNX-backed embedder (`bge-small-en-v1.5` via FastEmbed) on the chunked course-lesson data from module 1.

## Setup

In [1]:
# %pip install fastembed sqlite-vec "psycopg[binary]" pgvector numpy

In [2]:
from vector_search import Embedder, InMemoryEngine, SQLiteEngine, PGVectorEngine, benchmark

## Load the corpus

Reuses the chunked lessons from module 1 (`gitsource` + `chunk_documents`). If you've already built `chunks` in another notebook, you can skip the download.

In [3]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner='DataTalksClub',
    repo_name='llm-zoomcamp',
    commit_id='8c1834d',
    allowed_extensions={'md'},
    filename_filter=lambda p: '/lessons/' in p,
)
files = reader.read()
documents = [f.parse() for f in files]
chunks = chunk_documents(documents, size=2000, step=1000)

# minsearch expects dicts with 'filename' and 'content'; our engines do too.
# Take a slice so this notebook runs in seconds, not minutes.
corpus = chunks[:120]
print(f'Working with {len(corpus)} chunks')
print('Example fields:', list(corpus[0].keys()))

Working with 120 chunks
Example fields: ['start', 'content', 'filename']


## Embeddings — what they actually look like

Before plumbing this into a database, let's see what an embedding *is*.

In [4]:
import numpy as np

embedder = Embedder('BAAI/bge-small-en-v1.5')
print(f'Model:     {embedder.model_name}')
print(f'Dimension: {embedder.dim}')

sample = embedder.embed_one('what is retrieval augmented generation?')
print(f'\nShape:  {sample.shape}')
print(f'dtype:  {sample.dtype}')
print(f'first 8 dims: {sample[:8]}')

Model:     BAAI/bge-small-en-v1.5
Dimension: 384

Shape:  (384,)
dtype:  float32
first 8 dims: [-0.03896343  0.01703334 -0.038042   -0.02390459  0.02411521  0.00358701
 -0.03798935  0.00568326]


### Sanity check: semantically close phrases should produce close vectors

If embeddings work, *"a dog chasing a ball"* should sit closer to *"a puppy playing fetch"* than to *"financial market analysis"*.

In [5]:
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

phrases = [
    'a dog chasing a ball',
    'a puppy playing fetch',
    'financial market analysis',
]
vecs = embedder.embed(phrases)

for i in range(len(phrases)):
    for j in range(i+1, len(phrases)):
        sim = cosine(vecs[i], vecs[j])
        print(f'sim({phrases[i]!r:<32} , {phrases[j]!r:<32}) = {sim:.3f}')

sim('a dog chasing a ball'           , 'a puppy playing fetch'         ) = 0.775
sim('a dog chasing a ball'           , 'financial market analysis'     ) = 0.428
sim('a puppy playing fetch'          , 'financial market analysis'     ) = 0.373


## Engine 1 — in-memory NumPy

The simplest thing that works: keep all the vectors in a matrix and use a dot product to search. No database, no extension, nothing to install. This is what you do first; you graduate to the others when memory or persistence becomes an issue.

In [6]:
inmem = InMemoryEngine(embedder=embedder)
inmem.add(corpus)

for r in inmem.search('how does the agentic loop work', k=3):
    print(f'{r.score:.3f}  {r.filename}')

0.812  01-agentic-rag/lessons/14-agentic-loop.md
0.780  01-agentic-rag/lessons/14-agentic-loop.md
0.768  01-agentic-rag/lessons/14-agentic-loop.md


## Engine 2 — sqlite-vec

Same idea, but persisted to a single `.db` file. Vectors live in a virtual table; queries use `MATCH` against the embedding column.

In [7]:
sqlite_engine = SQLiteEngine(embedder=embedder, db_path=':memory:')
sqlite_engine.add(corpus)

for r in sqlite_engine.search('how does the agentic loop work', k=3):
    print(f'{r.score:.3f}  {r.filename}')

-0.613  01-agentic-rag/lessons/14-agentic-loop.md
-0.663  01-agentic-rag/lessons/14-agentic-loop.md
-0.681  01-agentic-rag/lessons/14-agentic-loop.md


## Engine 3 — pgvector (optional)

Requires a running Postgres with the `pgvector` extension. Easiest way:

```bash
docker run -d --name pgvector \
    -e POSTGRES_PASSWORD=postgres \
    -p 5432:5432 \
    pgvector/pgvector:pg16
```

If the container isn't up, skip this cell — the other two engines already make the point.

In [9]:
try:
    pg_engine = PGVectorEngine(embedder=embedder)
    pg_engine.add(corpus)
    for r in pg_engine.search('how does the agentic loop work', k=3):
        print(f'{r.score:.3f}  {r.filename}')
except Exception as e:
    pg_engine = None
    print(f'pgvector not available — skipping. ({type(e).__name__}: {e})')

0.812  01-agentic-rag/lessons/14-agentic-loop.md
0.780  01-agentic-rag/lessons/14-agentic-loop.md
0.768  01-agentic-rag/lessons/14-agentic-loop.md


## Head-to-head

Same query across all available engines, side by side. The engines should largely agree on the top result — the embedder does the heavy lifting; storage is mostly a logistics choice.

In [10]:
engines = [inmem, sqlite_engine]
if pg_engine is not None:
    engines.append(pg_engine)

queries = [
    'how does the agentic loop work',
    'what is the difference between chunking strategies',
    'how to evaluate retrieval quality',
]

results = benchmark(engines, corpus, queries, k=3)

import pandas as pd
df = pd.DataFrame(results)
df

[in-memory (numpy)] indexed 120 docs in 15.16s
[sqlite-vec] indexed 120 docs in 15.38s
[pgvector] indexed 120 docs in 15.25s


,engine,query,top_filename,top_score,latency_ms
0,in-memory (numpy),how does the agentic loop work,01-agentic-rag/lessons/14-agentic-loop.md,0.812241,6.926467
1,sqlite-vec,how does the agentic loop work,01-agentic-rag/lessons/14-agentic-loop.md,-0.612796,6.681892
2,pgvector,how does the agentic loop work,01-agentic-rag/lessons/14-agentic-loop.md,0.812241,7.927325
3,in-memory (numpy),what is the difference between chunking strate...,01-agentic-rag/lessons/09-data-ingestion.md,0.692678,5.650164
4,sqlite-vec,what is the difference between chunking strate...,01-agentic-rag/lessons/09-data-ingestion.md,-0.783992,5.608609
5,pgvector,what is the difference between chunking strate...,01-agentic-rag/lessons/09-data-ingestion.md,0.692678,6.975144
6,in-memory (numpy),how to evaluate retrieval quality,02-vector-search/lessons/10-next-steps.md,0.711455,5.774826
7,sqlite-vec,how to evaluate retrieval quality,02-vector-search/lessons/10-next-steps.md,-0.759664,5.486322
8,pgvector,how to evaluate retrieval quality,02-vector-search/lessons/10-next-steps.md,0.711455,7.553339


## What I took away

- **Embeddings are the *interesting* part; storage is plumbing.** Swap the database, same model, same data → same neighbours. Picking a vector DB is an ops decision, not an ML one.
- **Cosine vs L2 vs inner-product** — the engines disagree on the default distance metric, so the *absolute* scores aren't comparable. The *ordering* is what matters, and it's stable.
- **sqlite-vec is shockingly underrated** for personal projects. A single file, no daemon, no Docker, full vector search.
- **FastEmbed (ONNX) means no GPU.** Local embeddings on CPU at a usable speed — huge for laptops, edge devices, and not-paying-OpenAI-for-embeddings.
- **Vector search isn't a silver bullet.** For exact-match queries (a filename, an ID, a literal phrase), keyword search still wins. The future is hybrid.

In [ ]:
# Cleanup
for eng in engines:
    eng.close()